# NS06 Tutorial A - Eigenvector Centrality

**Prerequisites:** graphs, adjacency matrices, and basic linear algebra.

**Learning goals**
- Understand **recursive importance** in the same language as the NS06 slides.
- Compute eigenvector centrality with **power iteration** and with **NetworkX**.
- Read convergence diagnostics computationally.
- Interpret eigenvector centrality on a real historical social network.
- Explain why pure recursive importance is brittle on directed acyclic graphs with **source nodes**.

**Outline**
1. Path-graph example from the slides.
2. Power iteration and convergence.
3. Florentine families as a real network example.
4. Directed acyclic graphs and the source-node problem.
5. Exercise and takeaway.

In [ ]:
from netsci_utils import *
import pandas as pd

set_seeds()

## 1. Recursive importance on the path graph $P_4$

In the lecture, eigenvector centrality is defined by

$$
x_i = \frac{1}{\kappa} \sum_j A_{ij} x_j
$$

A node is important if it is connected to important nodes.

On the path graph $P_4$, the two middle nodes should outrank the two endpoints.

In [ ]:
P4 = nx.path_graph([1, 2, 3, 4])
pos_path = {1: (0, 0), 2: (1, 0), 3: (2, 0), 4: (3, 0)}

degree_path = dict(P4.degree())
eig_path = nx.eigenvector_centrality(P4, max_iter=1000)

path_table = pd.DataFrame({
    'node': [1, 2, 3, 4],
    'degree': [degree_path[n] for n in [1, 2, 3, 4]],
    'eigenvector': [eig_path[n] for n in [1, 2, 3, 4]],
})

plot_metric(
    P4,
    eig_path,
    title='Path graph P4: size / color = eigenvector centrality',
    pos=pos_path,
    figure_size=FIGURE_SIZE_SMALL,
    with_labels=True,
    min_node_size_px=28,
    max_node_size_px=48,
)

print(path_table.round(6).to_string(index=False))
print(f"\ncentral / endpoint ratio = {eig_path[2] / eig_path[1]:.3f}")

The ranking matches the lecture story:

- nodes 2 and 3 tie for first place
- nodes 1 and 4 tie for second place

Degree gives the same qualitative order here, but eigenvector centrality adds a quantitative distinction that depends on the whole graph structure.

## 2. Power iteration and convergence

Computationally, eigenvector centrality is often obtained by **power iteration**:

1. start from a positive vector
2. multiply by the adjacency matrix
3. renormalize
4. repeat until the vector stabilizes

This makes the Perron-Frobenius picture operational.

In [ ]:
def power_iteration_eigenvector(G, nodelist=None, max_iter=200, tol=1e-10):
    if nodelist is None:
        nodelist = list(G.nodes())
    A = nx.to_numpy_array(G, nodelist=nodelist, dtype=float)
    x = np.ones(len(nodelist), dtype=float)
    x /= np.linalg.norm(x)

    history = []
    for iteration in range(max_iter):
        x_new = A @ x
        norm = np.linalg.norm(x_new)
        if norm == 0:
            raise ValueError('Power iteration reached the zero vector.')
        x_new /= norm
        kappa = float(x_new @ (A @ x_new))
        residual = float(np.linalg.norm(A @ x_new - kappa * x_new))
        history.append({
            'iteration': iteration + 1,
            'kappa': kappa,
            'residual': residual,
            **{str(node): value for node, value in zip(nodelist, x_new)},
        })
        if np.linalg.norm(x_new - x) < tol:
            x = x_new
            break
        x = x_new

    return dict(zip(nodelist, x)), pd.DataFrame(history)


eig_manual, diagnostics = power_iteration_eigenvector(P4, nodelist=[1, 2, 3, 4])
compare = pd.DataFrame({
    'node': [1, 2, 3, 4],
    'manual': [eig_manual[n] for n in [1, 2, 3, 4]],
    'networkx': [eig_path[n] for n in [1, 2, 3, 4]],
})
compare['abs_diff'] = (compare['manual'] - compare['networkx']).abs()

print('First iterations of power iteration:')
print(diagnostics.head(6).round(6).to_string(index=False))
print('\nManual vs NetworkX:')
print(compare.round(6).to_string(index=False))

fig, ax = plt.subplots(figsize=FIGURE_SIZE_SMALL)
ax.plot(diagnostics['iteration'], diagnostics['residual'], marker='o', color=NODE_COLOR, linewidth=2)
style_axis(ax, title='Power-iteration residual on P4', xlabel='iteration', ylabel='||Ax - kappa x||', yscale='log')
plt.show()

**Computational interpretation.**

The residual $\|Ax - \kappa x\|$ measures how close the current vector is to a true eigenvector.

This is the quantity to monitor in practice. A stable ranking is not enough; we also want the eigenvector equation to be nearly satisfied.

## 3. Real network example: Florentine families

This graph records marriage and business ties among prominent families in Renaissance Florence.

Eigenvector centrality is useful here because status depends not only on how many ties a family has, but on whether those ties point into the influential core.

In [ ]:
families = nx.florentine_families_graph()
pos_families = nx.spring_layout(families, seed=RANDOM_SEED)

degree_families = nx.degree_centrality(families)
eig_families = nx.eigenvector_centrality(families, max_iter=1000)
family_table = pd.DataFrame({
    'family': list(families.nodes()),
    'degree': [degree_families[n] for n in families.nodes()],
    'eigenvector': [eig_families[n] for n in families.nodes()],
}).sort_values('eigenvector', ascending=False)

plot_metric(
    families,
    eig_families,
    title='Florentine families: size / color = eigenvector centrality',
    pos=pos_families,
    with_labels=True,
    font_size=8,
    min_node_size_px=26,
    max_node_size_px=58,
)

fig, ax = plt.subplots(figsize=FIGURE_SIZE_SMALL)
ax.scatter(family_table['degree'], family_table['eigenvector'], color=NODE_COLOR, s=55)
for family in ['Medici', 'Strozzi', 'Guadagni', 'Albizzi']:
    row = family_table.loc[family_table['family'] == family].iloc[0]
    ax.text(row['degree'] + 0.002, row['eigenvector'], family, fontsize=8)
style_axis(ax, title='Degree vs eigenvector centrality', xlabel='degree centrality', ylabel='eigenvector centrality')
plt.show()

print(family_table.head(10).round(4).to_string(index=False))

**Real-world implication.**

A family can have moderate degree and still score highly if it is connected to the right families.

That is exactly why eigenvector centrality is useful in alliance, prestige, and influence networks: it measures **who is connected to whom**, not just **how many** ties exist.

## 4. Why directed acyclic graphs are problematic

In the directed setting discussed in NS06, recursive importance comes from **incoming neighbors**.

On a directed acyclic graph, pure recursive support can collapse toward sinks because source nodes have no incoming support to propagate.

In [ ]:
dag = nx.DiGraph([('A', 'B'), ('B', 'C')])
order = ['A', 'B', 'C']
influence = nx.to_numpy_array(dag, nodelist=order, dtype=float).T

x = np.ones(len(order))
cascade_rows = []
for step in range(4):
    cascade_rows.append({'step': step, **{node: x[i] for i, node in enumerate(order)}})
    x = influence @ x
cascade = pd.DataFrame(cascade_rows)

pos_dag = {'A': (0, 0), 'B': (1, 0), 'C': (2, 0)}
plot_graph(
    dag,
    title='Directed chain A -> B -> C',
    pos=pos_dag,
    figure_size=FIGURE_SIZE_SMALL,
    with_labels=True,
    node_size=1400,
    arrows=True,
    arrowsize=18,
)

print('Iterating x^(t+1) = A^T x^(t):')
print(cascade.to_string(index=False))
print('\nNetworkX eigenvector centrality on the same DAG:')
print(pd.Series(nx.eigenvector_centrality(dag, max_iter=1000)).round(6).to_string())

The computational issue is structural, not numerical:

- source nodes have no incoming support
- sinks absorb recursive mass
- on a DAG, pure eigenvector centrality is a fragile prestige model

This is why NS06 introduces **Katz centrality** next.

## 5. Exercise

Add the edge `C -> A` to turn the chain into a directed cycle.

Before running the next cell, predict:

- does the zero-centrality cascade disappear?
- should the three nodes now tie?

In [ ]:
cycle = dag.copy()
cycle.add_edge('C', 'A')
cycle_eig = nx.eigenvector_centrality(cycle, max_iter=1000)
print(pd.Series(cycle_eig).round(6).to_string())

## Takeaway

- **Eigenvector centrality** measures recursive importance.
- **Power iteration** is the computational route from the definition to the ranking.
- On real networks, recursive importance can differ substantially from raw degree.
- On directed acyclic graphs, pure recursive support is brittle because of **source nodes**.